In [1]:
print("Hello, world!")

Hello, world!


In [ ]:
import jax.numpy as jnp
from flax import nnx

class ResNetBlock(nnx.Module):
    def __init__(self, channels: int, rngs: nnx.Rngs):
        # Keeps channels constant to allow direct addition
        self.conv1 = nnx.Conv(channels, channels, kernel_size=(3, 3), padding='SAME', rngs=rngs)
        self.bn1 = nnx.BatchNorm(channels, rngs=rngs)
        self.conv2 = nnx.Conv(channels, channels, kernel_size=(3, 3), padding='SAME', rngs=rngs)
        self.bn2 = nnx.BatchNorm(channels, rngs=rngs)

    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        residual = x
        
        # Main processing branch
        y = nnx.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        
        # Element-wise addition: reuses existing memory shape
        return nnx.relu(y + residual)

In [ ]:
class DenseNetBlock(nnx.Module):
    def __init__(self, in_channels: int, growth_rate: int, rngs: nnx.Rngs):
        self.bn1 = nnx.BatchNorm(in_channels, rngs=rngs)
        # Only generates a small number of new features (growth_rate)
        self.conv1 = nnx.Conv(in_channels, growth_rate, kernel_size=(3, 3), padding='SAME', rngs=rngs)

    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        # Bottleneck/Processing step
        y = nnx.relu(self.bn1(x))
        new_features = self.conv1(y)
        
        # Concatenation: forces allocation of a new, larger tensor block
        return jnp.concatenate([x, new_features], axis=-1)

In [ ]:
import time
import jax
import jax.numpy as jnp
from flax import nnx

# 1. Setup dimensions
B, H, W, C = 32, 64, 64, 64
growth_rate = 32
num_warmups = 10
num_repeats = 100

# Initialize keys and input data
rngs = nnx.Rngs(42)
x = jnp.ones((B, H, W, C))

# Initialize our modules
resnet_block = ResNetBlock(channels=C, rngs=rngs)
densenet_block = DenseNetBlock(in_channels=C, growth_rate=growth_rate, rngs=rngs)

# 2. Define JIT-compiled execution functions
@nnx.jit
def profile_resnet(tensor):
    return resnet_block(tensor)

@nnx.jit
def profile_densenet(tensor):
    return densenet_block(tensor)

def run_benchmark(name, jit_fn, init_tensor):
    print(f"\n--- Profiling {name} ---")
    
    # --- COLD RUN (Measures Compilation + First Execution) ---
    start = time.perf_counter()
    out = jit_fn(init_tensor)
    out.block_until_ready()  # Force synchronous wait
    cold_time = time.perf_counter() - start
    print(f"Cold Run (Compile Time included): {cold_time:.4f} seconds")
    
    # Warmup steps to clear any caching artifacts
    for _ in range(num_warmups):
        jit_fn(init_tensor).block_until_ready()
        
    # --- WARM RUNS (Measures Pure Execution Speed) ---
    start = time.perf_counter()
    for _ in range(num_repeats):
        jit_fn(init_tensor).block_until_ready()
    warm_time = (time.perf_counter() - start) / num_repeats
    print(f"Warm Run Average (Pure Execution):  {warm_time * 1000:.4f} ms")

# Run both benchmarks
run_benchmark("ResNet Block (Addition)", profile_resnet, x)
run_benchmark("DenseNet Block (Concatenation)", profile_densenet, x)